# Supermarket Sales Dashboard
**Project 2 – Data Analytics using Python and Jupyter Notebook**

## Objective
Analyze supermarket sales data to understand sales performance by city, branch, product line, customer type, gender, payment method, and month. Create visualizations and business insights.

**Dataset:** `SuperMarket Analysis.csv`

**Libraries:** Pandas, Matplotlib, Seaborn.

## 1. Import Libraries
Pandas is used to load, clean, and analyze tabular data. Matplotlib and Seaborn are used to create charts.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

## 2. Load the Dataset
Keep `SuperMarket Analysis.csv` in the same folder as this notebook. `read_csv()` loads the file into a Pandas DataFrame named `df`.

In [ ]:
df = pd.read_csv("SuperMarket Analysis.csv")
print("Dataset loaded successfully!")
display(df.head())

## 3. Inspect the Dataset
`shape` gives the number of rows and columns, `info()` shows data types and non-null counts, and `isnull().sum()` counts missing values.

In [ ]:
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nMissing values:")
display(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

## 4. Data Cleaning
Remove duplicate records, trim extra spaces in text columns, and convert the Date and Time columns into suitable formats. The supplied dataset currently has no missing values; this check is retained so the notebook can verify the data when rerun.

In [ ]:
df = df.drop_duplicates().copy()

text_columns = df.select_dtypes(include="object").columns
for col in text_columns:
    df[col] = df[col].str.strip()

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["Time"] = pd.to_datetime(df["Time"], format="%I:%M:%S %p", errors="coerce").dt.time

numeric_columns = [
    "Unit price", "Quantity", "Tax 5%", "Sales",
    "cogs", "gross margin percentage", "gross income", "Rating"
]
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Missing values after type conversion:")
display(df.isnull().sum())

## 5. Data Processing
Create useful time-based columns. `Month` supports monthly sales analysis and `Hour` can be used to study busy shopping times. Sales is already provided in the dataset, so we use it rather than calculating it again.

In [ ]:
df["Month"] = df["Date"].dt.to_period("M").astype(str)
df["Hour"] = pd.to_datetime(df["Time"].astype(str), format="%H:%M:%S", errors="coerce").dt.hour

print("Processed dataset:")
display(df.head())

## 6. Key Performance Indicators (KPIs)
Calculate total sales, total gross income, number of invoices, average customer rating, and total quantity sold.

In [ ]:
total_sales = df["Sales"].sum()
total_gross_income = df["gross income"].sum()
total_invoices = df["Invoice ID"].nunique()
average_rating = df["Rating"].mean()
total_quantity = df["Quantity"].sum()

print(f"Total Sales: {total_sales:,.2f}")
print(f"Total Gross Income: {total_gross_income:,.2f}")
print(f"Number of Invoices: {total_invoices:,}")
print(f"Average Customer Rating: {average_rating:.2f} / 10")
print(f"Total Quantity Sold: {total_quantity:,}")

## 7. Sales Analysis
Group sales by city, branch, product line, customer type, gender, and payment method. `groupby()` creates groups and `sum()` totals sales in each group.

In [ ]:
city_sales = df.groupby("City")["Sales"].sum().sort_values(ascending=False)
branch_sales = df.groupby("Branch")["Sales"].sum().sort_values(ascending=False)
product_sales = df.groupby("Product line")["Sales"].sum().sort_values(ascending=False)
customer_sales = df.groupby("Customer type")["Sales"].sum().sort_values(ascending=False)
gender_sales = df.groupby("Gender")["Sales"].sum().sort_values(ascending=False)
payment_sales = df.groupby("Payment")["Sales"].sum().sort_values(ascending=False)

print("Sales by City:")
display(city_sales.to_frame("Total Sales"))

print("Sales by Branch:")
display(branch_sales.to_frame("Total Sales"))

print("Sales by Product Line:")
display(product_sales.to_frame("Total Sales"))

print("Sales by Customer Type:")
display(customer_sales.to_frame("Total Sales"))

print("Sales by Gender:")
display(gender_sales.to_frame("Total Sales"))

print("Sales by Payment Method:")
display(payment_sales.to_frame("Total Sales"))

## 8. Monthly Sales Analysis
Group sales by month to understand how total sales change over time.

In [ ]:
monthly_sales = df.groupby("Month")["Sales"].sum().sort_index()
display(monthly_sales.to_frame("Total Sales"))

print("Highest sales month:", monthly_sales.idxmax())
print("Lowest sales month:", monthly_sales.idxmin())

## 9. Data Visualizations
The following charts show sales by city, product line, monthly trend, and payment method. Each chart is also saved as a PNG image.

In [ ]:
# Chart 1: Sales by City
city_sales.plot(kind="bar", title="Total Sales by City")
plt.xlabel("City")
plt.ylabel("Sales")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("sales_by_city.png", dpi=150)
plt.show()

In [ ]:
# Chart 2: Sales by Product Line
product_sales.sort_values().plot(kind="barh", title="Total Sales by Product Line")
plt.xlabel("Sales")
plt.ylabel("Product Line")
plt.tight_layout()
plt.savefig("sales_by_product_line.png", dpi=150)
plt.show()

In [ ]:
# Chart 3: Monthly Sales Trend
monthly_sales.plot(kind="line", marker="o", title="Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("monthly_sales_trend.png", dpi=150)
plt.show()

In [ ]:
# Chart 4: Sales by Payment Method
payment_sales.plot(kind="pie", autopct="%1.1f%%", startangle=90,
                   title="Sales Contribution by Payment Method")
plt.ylabel("")
plt.tight_layout()
plt.savefig("sales_by_payment_method.png", dpi=150)
plt.show()

In [ ]:
# Chart 5: Customer Rating by Product Line
sns.barplot(data=df, x="Product line", y="Rating", estimator="mean", errorbar=None)
plt.title("Average Customer Rating by Product Line")
plt.xlabel("Product Line")
plt.ylabel("Average Rating")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig("rating_by_product_line.png", dpi=150)
plt.show()

## 10. Business Insights
Generate data-based findings from the analysis. These statements use the dataset's actual calculated results.

In [ ]:
print("BUSINESS INSIGHTS")
print("1. Highest sales city:", city_sales.idxmax(), f"({city_sales.max():,.2f})")
print("2. Highest sales branch:", branch_sales.idxmax(), f"({branch_sales.max():,.2f})")
print("3. Top product line:", product_sales.idxmax(), f"({product_sales.max():,.2f})")
print("4. Highest sales month:", monthly_sales.idxmax(), f"({monthly_sales.max():,.2f})")
print("5. Most-used payment method by transaction count:", df["Payment"].value_counts().idxmax())
print(f"6. Average customer rating: {average_rating:.2f} out of 10")

## 11. Business Recommendations
1. Review the strongest product lines and maintain sufficient stock.
2. Investigate lower-sales product lines and test targeted promotions.
3. Compare customer ratings across product lines and improve service where ratings are weaker.
4. Keep payment options convenient because customers use multiple payment methods.
5. Monitor monthly sales to plan inventory and promotional campaigns.

Recommendations should be reviewed alongside costs, profit, and stock availability before decisions are made.

## 12. Save the Cleaned Dataset
Save the cleaned and processed dataset as a new CSV file. The original dataset is not overwritten.

In [ ]:
df.to_csv("cleaned_supermarket_sales.csv", index=False)
print("Cleaned dataset saved as cleaned_supermarket_sales.csv")

## 13. Conclusion
This project used Python to inspect, clean, process, and analyze supermarket sales data. It calculated key performance indicators, compared sales across business dimensions, visualized trends, and generated business insights and recommendations.